# Experiment 4: Data Augmentation --- RandomErasing + RandomAffine

## Rationale

Experiments 2 (architecture) and 3 (Focal Loss) both failed to improve Shirt TPR. Augmentation that breaks silhouette-only shortcuts forces the model to use residual texture cues.

**Single variable changed**: training transform --- added RandomAffine(plusmn5deg, plusmn5%) + RandomErasing(p=0.3)
**Held constant**: architecture (DiagnosticCNN), loss (CrossEntropy), optimizer (Adam lr=0.001), epochs (15), test transform

| Step | Description | What it does | Import path |
|------|-------------|--------------|-------------|
| 1 | Import Libraries | Load PyTorch, src modules, detect device | --- |
| 2 | Load Dataset with Augmentation | Apply RandomAffine + RandomErasing to training set | `src/data_utils.py` |
| 3 | Define DiagnosticCNN | Identical architecture to E1 baseline | --- |
| 4 | Train with Augmentation | Train 15 epochs on augmented data | `src/train_utils.py` |
| 5 | Evaluate Model | Per-class TPR, Precision, confusion matrix | `src/eval_utils.py` |
| 6 | ROC & PR Curves | ROC-AUC and PR-AUC scores | `src/eval_utils.py`, `src/vis_utils.py` |
| 7 | Compare with E1 | Side-by-side metrics vs no-augmentation baseline | `src/eval_utils.py` |
| 8 | Save Outputs | Save metrics to outputs/error_analysis/augmentation/ | --- |

---


In [ ]:
import os,sys
# Detect project root: look for src/ directory in CWD or parents
def _find_root(marker="src", max_up=3):
    p = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.getcwd())
PROJ_ROOT = _find_root()
sys.path.insert(0, PROJ_ROOT)

import os, torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms

from src.data_utils import load_fashionmnist, get_dataloaders
from src.train_utils import train_one_epoch
from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores)

OUT_DIR = '../outputs/error_analysis/augmentation'
os.makedirs(OUT_DIR, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
if torch.backends.mps.is_available():    device = 'mps'
elif torch.cuda.is_available():          device = 'cuda'
else:                                    device = 'cpu'
print(f"Device: {device}")

PyTorch: 2.13.0+cu130
Device: cuda


## Dataset — single variable change in transforms

In [2]:
# Standard transform (identical to E1)
# Used only for test set
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Augmented transform — the ONLY change from E1
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15), value=0),
    transforms.Normalize((0.5,), (0.5,)),
])

train_dataset = __import__('torchvision').datasets.FashionMNIST(
    root='../data', train=True, download=True, transform=train_transform
)
test_dataset = __import__('torchvision').datasets.FashionMNIST(
    root='../data', train=False, download=True, transform=base_transform
)

class_names = train_dataset.classes
train_loader, test_loader = get_dataloaders(train_dataset, test_dataset, batch_size=64)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")
print(f"Train transform: {train_transform}")

Train batches: 938, Test batches: 157
Train transform: Compose(
    ToTensor()
    RandomAffine(degrees=[-5.0, 5.0], translate=(0.05, 0.05))
    RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0, inplace=False)
    Normalize(mean=(0.5,), std=(0.5,))
)


## Architecture — identical to E1 DiagnosticCNN

In [3]:
class DiagnosticCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, num_classes)
        self.relu = nn.ReLU(inplace=True)

    def get_features(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.pool3(x)
        x = self.global_pool(x)
        return x.view(x.size(0), -1)

    def forward(self, x):
        x = self.get_features(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x


model = DiagnosticCNN().to(device)
print(f"DiagnosticCNN params: {sum(p.numel() for p in model.parameters()):,}")

DiagnosticCNN params: 140,778


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


## Training — identical to E1 (CrossEntropyLoss, Adam, 15 epochs)

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 15

train_losses = []
model.train()
for epoch in range(num_epochs):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss:.4f}')

with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:
    for loss in train_losses:
        f.write(f'{loss}\n')
print(f"Losses saved.")

Epoch [1/15], Loss: 0.5706
Epoch [2/15], Loss: 0.3712
Epoch [3/15], Loss: 0.3241
Epoch [4/15], Loss: 0.2964
Epoch [5/15], Loss: 0.2795
Epoch [6/15], Loss: 0.2656
Epoch [7/15], Loss: 0.2543
Epoch [8/15], Loss: 0.2449
Epoch [9/15], Loss: 0.2361
Epoch [10/15], Loss: 0.2317
Epoch [11/15], Loss: 0.2248
Epoch [12/15], Loss: 0.2217
Epoch [13/15], Loss: 0.2139
Epoch [14/15], Loss: 0.2104
Epoch [15/15], Loss: 0.2045
Losses saved.


## Evaluation — identical pipeline

In [5]:
accuracy, cm, per_class = evaluate_detailed(model, test_loader, device, class_names, model_name='AugCNN')
probas, labels = get_all_probas_and_labels(model, test_loader, device, 10)
roc_scores = compute_roc_auc_scores(probas, labels, model_name='AugCNN')
pr_scores = compute_pr_auc_scores(probas, labels, model_name='AugCNN')

with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'Test Accuracy (fraction): {accuracy / 100:.4f}\n\n')
    f.write(f'Macro ROC-AUC: {roc_scores["macro"]:.6f}\n')
    f.write(f'Macro PR-AUC:  {pr_scores["macro"]:.6f}\n\n')
    f.write(f'{"Class":<15} {"ROC-AUC":>10} {"PR-AUC":>10} {"TPR":>10} {"Precision":>10}\n')
    f.write('-' * 55 + '\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']
        prec = per_class[name]['Precision']
        f.write(f'{name:<15} {roc_scores[f"class_{i}"]:>10.4f} {pr_scores[f"class_{i}"]:>10.4f} {tpr:>10.4f} {prec:>10.4f}\n')

cm_np = cm.cpu().numpy()
with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names:
        f.write(f'{name:>15}')
    f.write('\n')
    for i in range(len(class_names)):
        f.write(f'{class_names[i]:>15}')
        for j in range(len(class_names)):
            f.write(f'{cm_np[i, j]:>15}')
        f.write('\n')

with open(os.path.join(OUT_DIR, 'misclassification_analysis.txt'), 'w') as f:
    f.write('Misclassification Analysis\n')
    f.write('=' * 70 + '\n\n')
    for c in range(len(class_names)):
        true_name = class_names[c]
        total_errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {true_name}  (errors: {total_errors})\n')
        f.write('-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0:
                continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')

print(f"\nAll results saved to {OUT_DIR}/")

  Test Accuracy: 92.45%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8340     0.0077     0.9236
  Trouser             0.9880     0.0016     0.9860
  Pullover            0.9360     0.0190     0.8455
  Dress               0.8880     0.0066     0.9377
  Coat                0.8820     0.0118     0.8927
  Sandal              0.9680     0.0004     0.9959
  Shirt               0.8100     0.0274     0.7663
  Sneaker             0.9860     0.0059     0.9490
  Bag                 0.9860     0.0012     0.9890
  Ankle boot          0.9670     0.0023     0.9787

All results saved to ../outputs/error_analysis/augmentation/


## Delta vs E1 Baseline

In [6]:
def load_e1_metrics(path):
    data = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5 and parts[0] != 'Class' and '-' not in line[:5]:
                cls, roc, pr, tpr, prec = parts[0], float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                data[cls] = {'tpr': tpr, 'precision': prec, 'pr_auc': pr}
    return data

e1 = load_e1_metrics('../outputs/error_analysis/metrics_summary.txt')

print(f'{"Class":<15} {"E1 TPR":>8} {"E4 TPR":>8} {"Δ TPR":>8} {"E1 Prec":>8} {"E4 Prec":>8} {"Δ Prec":>8}')
print('-' * 63)
for name in class_names:
    if name in e1:
        e1_tpr = e1[name]['tpr']
        e1_prec = e1[name]['precision']
        e4_tpr = per_class[name]['TPR']
        e4_prec = per_class[name]['Precision']
        print(f'{name:<15} {e1_tpr:>8.3f} {e4_tpr:>8.3f} {e4_tpr - e1_tpr:>+8.3f} {e1_prec:>8.3f} {e4_prec:>8.3f} {e4_prec - e1_prec:>+8.3f}')

print(f'\nAccuracy:  E1=92.50%  E4={accuracy:.2f}%  Δ={accuracy - 92.50:+.2f}%')
print(f'Macro PR:   E1=0.9712  E4={pr_scores["macro"]:.4f}  Δ={pr_scores["macro"] - 0.9712:+.4f}')

# Also compare Shirt sink specifically
e1_shirt_err = 153 + 163  # E1 Shirt + T-shirt errors
e4_shirt_err = cm_np[class_names.index('Shirt')].sum() - cm_np[class_names.index('Shirt'), class_names.index('Shirt')]
e4_tshirt_err = cm_np[class_names.index('T-shirt/top')].sum() - cm_np[class_names.index('T-shirt/top'), class_names.index('T-shirt/top')]
print(f'\nUpper-body errors: E1=649, E4={e4_shirt_err + e4_tshirt_err}')
print(f'  Shirt errors:    E1=153, E4={e4_shirt_err}')
print(f'  T-shirt errors:  E1=163, E4={e4_tshirt_err}')

Class             E1 TPR   E4 TPR    Δ TPR  E1 Prec  E4 Prec   Δ Prec
---------------------------------------------------------------
Trouser            0.988    0.988   +0.000    0.990    0.986   -0.004
Pullover           0.890    0.936   +0.046    0.896    0.846   -0.051
Dress              0.907    0.888   -0.019    0.940    0.938   -0.002
Coat               0.870    0.882   +0.012    0.931    0.893   -0.039
Sandal             0.978    0.968   -0.010    0.990    0.996   +0.006
Shirt              0.847    0.810   -0.037    0.723    0.766   +0.044
Sneaker            0.991    0.986   -0.005    0.946    0.949   +0.003
Bag                0.986    0.986   +0.000    0.982    0.989   +0.007

Accuracy:  E1=92.50%  E4=92.45%  Δ=-0.05%
Macro PR:   E1=0.9712  E4=0.9719  Δ=+0.0007

Upper-body errors: E1=649, E4=356
  Shirt errors:    E1=153, E4=190
  T-shirt errors:  E1=163, E4=166
